In [1]:
import pandas as pd

df = pd.read_csv(
    "/content/02_hidden_llm_benchmark_15000.csv"
)

print(df.shape)
print(df.columns.tolist())

(15000, 10)
['interaction_id', 'crm_note_id', 'reference_sentiment', 'reference_topics', 'reference_objections', 'reference_primary_objection', 'reference_objection_severity', 'reference_aspect_sentiments', 'split_assignment', 'reference_source']


In [2]:
#Check the splits
print(df["split_assignment"].value_counts())

split_assignment
train             10546
validation         1521
phrase_holdout     1471
final_hidden       1462
Name: count, dtype: int64


In [3]:
#Define our text and labels
TEXT_COL = "crm_note_id"
TARGET_COL = "reference_primary_objection"
SENTIMENT_COL = "reference_sentiment"

In [4]:
#Load your 15K preprocessed CRM data
crm = pd.read_csv(
    "/content/CTS_CRM_15000_NLP_Preprocessed.csv"
)

print(crm.shape)
print(crm.columns.tolist())

(6468, 23)
['interaction_id', 'crm_note_id', 'interaction_date', 'rep_id', 'hcp_id', 'city', 'region', 'territory_id', 'hcp_specialization', 'therapeutic_area', 'drug_id', 'drug_name', 'brand_name', 'crm_note', 'word_count_raw', 'text_raw', 'text_lower', 'text_normalized', 'text_no_admin', 'text_no_punct', 'text_no_stopwords', 'clean_text', 'clean_word_count']


In [5]:
TEXT_COL = "clean_text"

In [6]:
#Merge the benchmark labels with your CRM text
data = crm.merge(
    df[
        [
            "interaction_id",
            "reference_sentiment",
            "reference_primary_objection",
            "split_assignment"
        ]
    ],
    on="interaction_id",
    how="inner"
)

print(data.shape)

(6468, 26)


In [7]:
print(data["split_assignment"].value_counts())

split_assignment
train             4603
validation         709
phrase_holdout     630
final_hidden       526
Name: count, dtype: int64


In [8]:
#DO NOT use the final hidden labels for training
train_data = data[
    data["split_assignment"] == "train"
].copy()

val_data = data[
    data["split_assignment"] == "validation"
].copy()

phrase_data = data[
    data["split_assignment"] == "phrase_holdout"
].copy()

hidden_data = data[
    data["split_assignment"] == "final_hidden"
].copy()

In [9]:
print(len(train_data))
print(len(val_data))
print(len(phrase_data))
print(len(hidden_data))

4603
709
630
526


In [10]:
#train text-only
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

In [11]:
#Word TF-IDF
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=100000
)

In [12]:
#Character TF-IDF
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 6),
    min_df=2,
    sublinear_tf=True,
    max_features=100000
)

In [13]:
#Transform training and validation
X_train_text = train_data["clean_text"].fillna("")
y_train = train_data["reference_primary_objection"]

X_val_text = val_data["clean_text"].fillna("")
y_val = val_data["reference_primary_objection"]

In [14]:
X_train_word = word_vectorizer.fit_transform(X_train_text)
X_val_word = word_vectorizer.transform(X_val_text)

X_train_char = char_vectorizer.fit_transform(X_train_text)
X_val_char = char_vectorizer.transform(X_val_text)

X_train = hstack([
    X_train_word,
    X_train_char
])

X_val = hstack([
    X_val_word,
    X_val_char
])

In [15]:
#Train the SVM
from sklearn.svm import LinearSVC

model = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=42
)

model.fit(
    X_train,
    y_train
)

LinearSVC(class_weight='balanced', random_state=42)

In [16]:
#Evaluate on validation
from sklearn.metrics import accuracy_score, f1_score

val_pred = model.predict(X_val)

val_accuracy = accuracy_score(
    y_val,
    val_pred
)

val_f1 = f1_score(
    y_val,
    val_pred,
    average="macro"
)

print("Validation Accuracy:", round(val_accuracy, 4))
print("Validation Macro-F1:", round(val_f1, 4))

Validation Accuracy: 0.969
Validation Macro-F1: 0.9609


In [17]:
#Tune C
C_values = [
    0.1,
    0.25,
    0.5,
    1,
    2,
    4,
    8,
    16
]
results = []

for C in C_values:

    model = LinearSVC(
        C=C,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    acc = accuracy_score(y_val, pred)

    f1 = f1_score(
        y_val,
        pred,
        average="macro"
    )

    results.append({
        "C": C,
        "accuracy": acc,
        "macro_f1": f1
    })

results_df = pd.DataFrame(results)

print(results_df)

       C  accuracy  macro_f1
0   0.10  0.963329  0.954912
1   0.25  0.967560  0.958737
2   0.50  0.966150  0.958607
3   1.00  0.968970  0.960869
4   2.00  0.970381  0.961136
5   4.00  0.968970  0.959502
6   8.00  0.968970  0.959502
7  16.00  0.968970  0.959502


In [18]:
#Test on phrase_holdout
best_C = results_df.loc[
    results_df["macro_f1"].idxmax(),
    "C"
]

final_model = LinearSVC(
    C=best_C,
    class_weight="balanced",
    random_state=42
)

final_model.fit(X_train, y_train)

LinearSVC(C=np.float64(2.0), class_weight='balanced', random_state=42)

In [19]:
#Transform phrase holdout:
X_phrase_word = word_vectorizer.transform(
    phrase_data["clean_text"].fillna("")
)

X_phrase_char = char_vectorizer.transform(
    phrase_data["clean_text"].fillna("")
)

X_phrase = hstack([
    X_phrase_word,
    X_phrase_char
])

In [20]:
phrase_pred = final_model.predict(X_phrase)

print(
    "Phrase Holdout Accuracy:",
    accuracy_score(
        phrase_data["reference_primary_objection"],
        phrase_pred
    )
)

print(
    "Phrase Holdout Macro-F1:",
    f1_score(
        phrase_data["reference_primary_objection"],
        phrase_pred,
        average="macro"
    )
)

Phrase Holdout Accuracy: 0.9507936507936507
Phrase Holdout Macro-F1: 0.9381223958856638


In [21]:
# Transform final hidden data

X_hidden_word = word_vectorizer.transform(
    hidden_data["clean_text"].fillna("")
)

X_hidden_char = char_vectorizer.transform(
    hidden_data["clean_text"].fillna("")
)

X_hidden = hstack([
    X_hidden_word,
    X_hidden_char
])

print("Hidden data shape:", X_hidden.shape)

Hidden data shape: (526, 17279)


In [22]:
hidden_pred = final_model.predict(X_hidden)

print("Predictions completed!")
print("Number of predictions:", len(hidden_pred))

Predictions completed!
Number of predictions: 526


In [23]:
#Calculate final accuracy
from sklearn.metrics import accuracy_score

hidden_accuracy = accuracy_score(
    hidden_data["reference_primary_objection"],
    hidden_pred
)

print(
    "FINAL HIDDEN ACCURACY:",
    round(hidden_accuracy, 4)
)

FINAL HIDDEN ACCURACY: 0.9696


In [24]:
#Calculate final Macro-F1
from sklearn.metrics import f1_score

hidden_macro_f1 = f1_score(
    hidden_data["reference_primary_objection"],
    hidden_pred,
    average="macro"
)

print(
    "FINAL HIDDEN MACRO-F1:",
    round(hidden_macro_f1, 4)
)

FINAL HIDDEN MACRO-F1: 0.9646


In [25]:
#classification report
from sklearn.metrics import classification_report

print(
    classification_report(
        hidden_data["reference_primary_objection"],
        hidden_pred
    )
)

                             precision    recall  f1-score   support

          Adherence Concern       0.97      1.00      0.99        37
      Clinical Evidence Gap       1.00      0.91      0.95        22
      Competitor Preference       0.97      1.00      0.98        32
          Dosing Complexity       0.90      0.97      0.93        29
           Efficacy Concern       1.00      0.94      0.97        47
       Eligibility Criteria       0.96      0.96      0.96        24
      Formulary Restriction       0.92      1.00      0.96        24
    High Out-of-Pocket Cost       1.00      0.94      0.97        48
               No Objection       0.99      1.00      1.00       116
          Poor Availability       1.00      0.97      0.98        33
        Prior Authorization       0.91      0.96      0.93        52
             Safety Concern       0.96      0.93      0.94        27
Side Effects / Tolerability       0.97      0.97      0.97        35

                   accuracy     

In [26]:
hidden_results = hidden_data[
    [
        "interaction_id",
        "clean_text",
        "reference_sentiment",
        "reference_primary_objection"
    ]
].copy()

hidden_results["predicted_objection"] = hidden_pred

hidden_results.to_csv(
    "/content/final_hidden_predictions.csv",
    index=False
)

print("Final hidden predictions saved!")

Final hidden predictions saved!


In [29]:
#on unseen data
from google.colab import files

uploaded = files.upload()

Saving Final_Unseen_CRM_Objection_Test_65.csv to Final_Unseen_CRM_Objection_Test_65.csv


In [30]:
import pandas as pd

unseen_df = pd.read_csv(
    "/content/Final_Unseen_CRM_Objection_Test_65.csv"
)

print(unseen_df.shape)
print(unseen_df.head())

(65, 3)
  test_id                                           crm_note  \
0    U001  The office is waiting for insurer authorizatio...   
1    U002  Treatment initiation has been postponed becaus...   
2    U003  The clinic needs assistance navigating a new a...   
3    U004  The physician says repeated authorization pape...   
4    U005  Several patients are waiting for payer approva...   

                                          clean_text  
0  the office is waiting for insurer authorizatio...  
1  treatment initiation has been postponed becaus...  
2  the clinic needs assistance navigating a new a...  
3  the physician says repeated authorization pape...  
4  several patients are waiting for payer approva...  


In [31]:
#Add your preprocessing imports
import re
import spacy
import nltk

from nltk.corpus import stopwords

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [32]:
stop_words = set(stopwords.words("english"))

words_to_keep = {
    "no",
    "not",
    "nor",
    "never",
    "neither",
    "without",
    "against"
}

stop_words = stop_words - words_to_keep

In [33]:
def normalize_abbreviations(text):

    text = re.sub(
        r'\bf\s*/\s*u\b',
        'follow up',
        text
    )

    text = re.sub(
        r'\bpa\b',
        'prior authorization',
        text
    )

    text = re.sub(
        r'\bre\b',
        'regarding',
        text
    )

    return text

In [34]:
def remove_admin_noise(text):

    # Remove representative IDs such as REP123
    text = re.sub(
        r'\brep\d+\b',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Remove dates such as Jan 15
    text = re.sub(
        r'\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+\d{1,2}\b',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Remove standalone numbers
    text = re.sub(
        r'\b\d+\b',
        ' ',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text

In [35]:
def remove_punctuation(text):

    text = re.sub(
        r'[^\w\s]',
        ' ',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text

In [36]:
def remove_stopwords(text):

    words = text.split()

    filtered_words = [
        word
        for word in words
        if word not in stop_words
    ]

    return " ".join(filtered_words)

In [37]:
nlp = spacy.load("en_core_web_sm")

In [38]:
#preprocessing function
def preprocess_for_model(text):

    if pd.isna(text):
        return ""

    # 1. Convert to string
    text = str(text)

    # 2. Lowercase
    text = text.lower()

    # 3. Normalize CRM abbreviations
    text = normalize_abbreviations(text)

    # 4. Remove administrative noise
    text = remove_admin_noise(text)

    # 5. Remove punctuation
    text = remove_punctuation(text)

    # 6. Remove stopwords
    text = remove_stopwords(text)

    # 7. Lemmatization
    doc = nlp(text)

    lemmas = []

    for token in doc:

        if not token.is_alpha:
            continue

        lemmas.append(
            token.lemma_.lower()
        )

    return " ".join(lemmas)

In [39]:
unseen_df["clean_text"] = (
    unseen_df["crm_note"]
    .fillna("")
    .apply(preprocess_for_model)
)

In [40]:
#result
display(
    unseen_df[
        ["test_id", "crm_note", "clean_text"]
    ].head(10)
)

,test_id,crm_note,clean_text
0,U001,The office is waiting for insurer authorizatio...,office wait insurer authorization physician st...
1,U002,Treatment initiation has been postponed becaus...,treatment initiation postpone payer not comple...
2,U003,The clinic needs assistance navigating a new a...,clinic need assistance navigate new authorizat...
3,U004,The physician says repeated authorization pape...,physician say repeat authorization paperwork s...
4,U005,Several patients are waiting for payer approva...,several patient wait payer approval receive pr...
5,U006,Patients are hesitant to continue because thei...,patient hesitant continue personal treatment e...
6,U007,The practice reports that the amount patients ...,practice report amount patient must pay affect...
7,U008,The physician is concerned that treatment expe...,physician concern treatment expense remain dif...
8,U009,Several patients have asked whether there are ...,several patient ask whether way reduce persona...
9,U010,The clinic sees affordability as a barrier for...,clinic see affordability barrier patient subst...


In [41]:
#Transform using your EXISTING vectorizers
X_unseen_word = word_vectorizer.transform(
    unseen_df["clean_text"].fillna("")
)

X_unseen_char = char_vectorizer.transform(
    unseen_df["clean_text"].fillna("")
)

In [42]:
#Combine features
from scipy.sparse import hstack

X_unseen = hstack([
    X_unseen_word,
    X_unseen_char
])

In [43]:
#Predict
unseen_pred = final_model.predict(X_unseen)

unseen_df["predicted_objection"] = unseen_pred

print(unseen_df[
    ["test_id", "crm_note", "predicted_objection"]
].to_string(index=False))

test_id                                                                                                        crm_note         predicted_objection
   U001                         The office is waiting for insurer authorization before the physician can start therapy.         Prior Authorization
   U002                     Treatment initiation has been postponed because the payer has not completed prior approval.         Prior Authorization
   U003                        The clinic needs assistance navigating a new authorization request from the health plan.         Prior Authorization
   U004                                The physician says repeated authorization paperwork is slowing treatment starts.         Prior Authorization
   U005                        Several patients are waiting for payer approval before receiving the prescribed therapy.         Prior Authorization
   U006                         Patients are hesitant to continue because their personal treatment expense is to

In [44]:
uploaded = files.upload()

Saving Final_Unseen_CRM_Objection_Answer_Key_65.csv to Final_Unseen_CRM_Objection_Answer_Key_65.csv


In [45]:
answer_key = pd.read_csv(
    "/content/Final_Unseen_CRM_Objection_Answer_Key_65.csv"
)

In [46]:
results = unseen_df.merge(
    answer_key,
    on="test_id",
    how="inner"
)

print(results.head())

  test_id                                           crm_note  \
0    U001  The office is waiting for insurer authorizatio...   
1    U002  Treatment initiation has been postponed becaus...   
2    U003  The clinic needs assistance navigating a new a...   
3    U004  The physician says repeated authorization pape...   
4    U005  Several patients are waiting for payer approva...   

                                          clean_text  predicted_objection  \
0  office wait insurer authorization physician st...  Prior Authorization   
1  treatment initiation postpone payer not comple...  Prior Authorization   
2  clinic need assistance navigate new authorizat...  Prior Authorization   
3  physician say repeat authorization paperwork s...  Prior Authorization   
4  several patient wait payer approval receive pr...  Prior Authorization   

          ground_truth  
0  Prior Authorization  
1  Prior Authorization  
2  Prior Authorization  
3  Prior Authorization  
4  Prior Authorization  


In [47]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(
    results["ground_truth"],
    results["predicted_objection"]
)

macro_f1 = f1_score(
    results["ground_truth"],
    results["predicted_objection"],
    average="macro"
)

print("Unseen Accuracy:", round(accuracy, 4))
print("Unseen Macro-F1:", round(macro_f1, 4))

Unseen Accuracy: 0.7692
Unseen Macro-F1: 0.7737


In [48]:
print(
    classification_report(
        results["ground_truth"],
        results["predicted_objection"]
    )
)

                             precision    recall  f1-score   support

          Adherence Concern       0.40      0.40      0.40         5
      Clinical Evidence Gap       1.00      0.80      0.89         5
      Competitor Preference       0.80      0.80      0.80         5
          Dosing Complexity       0.83      1.00      0.91         5
           Efficacy Concern       1.00      0.60      0.75         5
       Eligibility Criteria       1.00      0.80      0.89         5
      Formulary Restriction       0.71      1.00      0.83         5
    High Out-of-Pocket Cost       0.75      0.60      0.67         5
               No Objection       0.57      0.80      0.67         5
          Poor Availability       1.00      0.80      0.89         5
        Prior Authorization       1.00      1.00      1.00         5
             Safety Concern       1.00      0.60      0.75         5
Side Effects / Tolerability       0.50      0.80      0.62         5

                   accuracy     

In [49]:
errors = results[
    results["ground_truth"] != results["predicted_objection"]
][
    [
        "test_id",
        "crm_note",
        "ground_truth",
        "predicted_objection"
    ]
]

print("Number of errors:", len(errors))
errors

Number of errors: 15


,test_id,crm_note,ground_truth,predicted_objection
5,U006,Patients are hesitant to continue because thei...,High Out-of-Pocket Cost,Side Effects / Tolerability
6,U007,The practice reports that the amount patients ...,High Out-of-Pocket Cost,Side Effects / Tolerability
13,U014,The clinic requested practical guidance for pa...,Side Effects / Tolerability,No Objection
17,U018,The practice questioned whether patients with ...,Efficacy Concern,High Out-of-Pocket Cost
19,U020,The HCP asked how consistently patients achiev...,Efficacy Concern,Adherence Concern
20,U021,The physician wants reassurance about potentia...,Safety Concern,Formulary Restriction
24,U025,The HCP would like additional reassurance that...,Safety Concern,No Objection
27,U028,Patients have had trouble obtaining the therap...,Poor Availability,Adherence Concern
32,U033,The physician asked for ideas to help patients...,Adherence Concern,Side Effects / Tolerability
33,U034,The clinic is seeing treatment interruptions c...,Adherence Concern,Dosing Complexity
